# Colab Experiment: Unlearnable Examples

Goal: generate class-wise error-minimizing noise on CIFAR-10, train a fresh victim on the protected train set, and evaluate on the clean test set.

## 1. Setup Colab / GitHub repo

This cell clones the repo when running from a fresh Colab runtime and installs dependencies.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
PROJECT_DIR = "adversarial-data-protection"

# If this notebook is opened directly in Colab, clone the GitHub repo first.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("src").exists():
    if not Path(PROJECT_DIR).exists():
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())
print("Installing: requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Optional Google Drive output

GitHub stores source code only. Use Drive if you want results to persist after the Colab runtime resets.

In [ ]:
# Google Drive dataset/results paths.
# Your Drive folder is: MyDrive/adversarial-data-protection/
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/adversarial-data-protection"
DRIVE_DATA_ROOT = f"{DRIVE_PROJECT_DIR}/data"
DRIVE_RESULTS_DIR = f"{DRIVE_PROJECT_DIR}/results"

DATA_ROOT = "./data"
RESULTS_ROOT = "./results"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_ROOT = DRIVE_DATA_ROOT
        RESULTS_ROOT = DRIVE_RESULTS_DIR
        Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
        Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
    except ImportError:
        print("Not running in Colab; using local ./data and ./results")

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)


## 3. Run experiment

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from scripts.run_experiment import setup_dirs, collect_clean_tensors, build_protected_tensors
from src.datasets import get_cifar10
from src.evaluation import compute_attack_success_rate, compute_linf, compute_psnr, compute_ssim
from src.models import evaluate, get_victim_resnet18, train_one_epoch
from src.techniques.unlearnable import generate_unlearnable_noise
from src.visualization import plot_before_after

# Medium benchmark defaults. The previous smoke config (1000 images, 2 epochs)
# left the clean baseline near random guessing, so it was not a valid benchmark.
SUBSET_SIZE = 5000
BATCH_SIZE = 128
EPSILON = 0.03
BASELINE_EPOCHS = 15
VICTIM_EPOCHS = 15
PGD_STEPS = 10
INNER_EPOCHS = 2
BASELINE_MIN_ACC = 0.40
LEARNING_RATE = 0.1
WEIGHT_DECAY = 5e-4
RUN_NAME = f"subset{SUBSET_SIZE}_eps{EPSILON}_base{BASELINE_EPOCHS}_victim{VICTIM_EPOCHS}_pgd{PGD_STEPS}_inner{INNER_EPOCHS}"
RUN_DIR = Path("results") / "unlearnable" / RUN_NAME
TABLE_DIR = RUN_DIR / "tables"
SAMPLE_DIR = RUN_DIR / "samples"
TENSOR_DIR = RUN_DIR / "tensors"
MODEL_DIR = RUN_DIR / "models"
FIGURE_DIR = RUN_DIR / "figures"
for path in [TABLE_DIR, SAMPLE_DIR, TENSOR_DIR, MODEL_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)


class CifarNormalize(nn.Module):
    def __init__(self, mean=CIFAR10_MEAN, std=CIFAR10_STD):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean.to(x.device, x.dtype)) / self.std.to(x.device, x.dtype)


def get_cifar_resnet18(device):
    """Victim/surrogate classifier with CIFAR-10 normalization inside the model."""
    return nn.Sequential(
        CifarNormalize(),
        get_victim_resnet18(num_classes=10, device="cpu"),
    ).to(device)


def train_classifier_strong(model_fn, loader, epochs, device, lr=LEARNING_RATE):
    """Train a CIFAR victim strongly enough that the clean baseline is meaningful."""
    model = model_fn().to(device)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=WEIGHT_DECAY,
        nesterov=True,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        loss, acc = train_one_epoch(model, loader, optimizer, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch + 1, "train_loss": loss, "train_accuracy": acc, "lr": round(current_lr, 6)})
        print(f"  epoch {epoch + 1}/{epochs} loss={loss} train_acc={acc} lr={current_lr:.6f}")
    return model, history


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
setup_dirs()

train_loader, test_loader = get_cifar10(root=DATA_ROOT, subset_size=SUBSET_SIZE, batch_size=BATCH_SIZE)
clean_x, clean_y = collect_clean_tensors(train_loader)

print("Training stronger clean baseline victim...")
baseline, baseline_history = train_classifier_strong(
    lambda: get_cifar_resnet18(device),
    train_loader,
    BASELINE_EPOCHS,
    device,
)
baseline_acc = evaluate(baseline, test_loader, device)
print("baseline_clean_test_accuracy:", baseline_acc)

if baseline_acc < BASELINE_MIN_ACC:
    print(
        f"WARNING: baseline accuracy {baseline_acc:.4f} is below {BASELINE_MIN_ACC:.2f}. "
        "The model still has not learned clean CIFAR-10 well enough; increase epochs/subset before using this as final evidence."
    )

print("Generating class-wise unlearnable noise...")
noise_dict = generate_unlearnable_noise(
    model_fn=lambda: get_cifar_resnet18(device),
    train_loader=train_loader,
    epsilon=EPSILON,
    pgd_steps=PGD_STEPS,
    inner_epochs=INNER_EPOCHS,
    device=device,
)
torch.save(noise_dict, TENSOR_DIR / "unlearnable_noise_dict.pt")
# Compatibility copy for Gradio demo.
torch.save(noise_dict, "results/unlearnable_noise_dict.pt")

protected_x = build_protected_tensors(
    "unlearnable", clean_x, clean_y, device, epsilon=EPSILON, noise_dict=noise_dict
)
torch.save(
    {
        "x_clean": clean_x.cpu(),
        "x_protected": protected_x.cpu(),
        "y": clean_y.cpu(),
        "epsilon": EPSILON,
        "technique": "unlearnable",
        "subset_size": SUBSET_SIZE,
        "run_name": RUN_NAME,
    },
    TENSOR_DIR / "protected_dataset.pt",
)
protected_loader = DataLoader(TensorDataset(protected_x, clean_y), batch_size=BATCH_SIZE, shuffle=True)

print("Training victim on protected train set with same baseline recipe...")
victim, victim_history = train_classifier_strong(
    lambda: get_cifar_resnet18(device),
    protected_loader,
    VICTIM_EPOCHS,
    device,
)
clean_acc, asr = compute_attack_success_rate(victim, test_loader, device)

torch.save(
    {
        "model_state_dict": baseline.state_dict(),
        "model": "ResNet-18+CIFAR-normalization",
        "dataset": "CIFAR-10",
        "train_data": "clean",
        "epochs": BASELINE_EPOCHS,
        "accuracy": baseline_acc,
        "run_name": RUN_NAME,
    },
    MODEL_DIR / "baseline_clean_model.pt",
)
torch.save(
    {
        "model_state_dict": victim.state_dict(),
        "model": "ResNet-18+CIFAR-normalization",
        "dataset": "CIFAR-10",
        "train_data": "protected",
        "epochs": VICTIM_EPOCHS,
        "accuracy": clean_acc,
        "run_name": RUN_NAME,
    },
    MODEL_DIR / "victim_protected_model.pt",
)

metrics = {
    "technique": "unlearnable",
    "victim_model": "ResNet-18+CIFAR-normalization",
    "subset_size": SUBSET_SIZE,
    "epsilon": EPSILON,
    "baseline_epochs": BASELINE_EPOCHS,
    "victim_epochs": VICTIM_EPOCHS,
    "pgd_steps": PGD_STEPS,
    "inner_epochs": INNER_EPOCHS,
    "baseline_clean_test_accuracy": baseline_acc,
    "protected_clean_test_accuracy": clean_acc,
    "accuracy_drop": round(baseline_acc - clean_acc, 4),
    "asr": asr,
    "psnr": compute_psnr(clean_x, protected_x),
    "ssim": compute_ssim(clean_x, protected_x),
    "linf": compute_linf(clean_x, protected_x),
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
}
print(metrics)

pd.DataFrame([metrics]).to_csv(TABLE_DIR / "unlearnable_experiment.csv", index=False)
pd.DataFrame(baseline_history).to_csv(TABLE_DIR / "baseline_training_history.csv", index=False)
pd.DataFrame(victim_history).to_csv(TABLE_DIR / "protected_training_history.csv", index=False)
# Compatibility copy for quick lookup across runs.
os.makedirs("results/tables", exist_ok=True)
pd.DataFrame([metrics]).to_csv("results/tables/unlearnable_experiment.csv", index=False)

baseline_df = pd.DataFrame(baseline_history)
victim_df = pd.DataFrame(victim_history)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(baseline_df["epoch"], baseline_df["train_accuracy"], marker="o", label="Clean baseline train")
ax.plot(victim_df["epoch"], victim_df["train_accuracy"], marker="s", label="Protected victim train")
ax.set_xlabel("Epoch")
ax.set_ylabel("Train accuracy")
ax.set_title("Training accuracy curves")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "training_accuracy_curves.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(baseline_df["epoch"], baseline_df["train_loss"], marker="o", label="Clean baseline train")
ax.plot(victim_df["epoch"], victim_df["train_loss"], marker="s", label="Protected victim train")
ax.set_xlabel("Epoch")
ax.set_ylabel("Train loss")
ax.set_title("Training loss curves")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "training_loss_curves.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Clean baseline", "Train on protected"]
values = [baseline_acc, clean_acc]
bars = ax.bar(labels, values, color=["#4C78A8", "#F58518"])
ax.set_ylabel("Clean test accuracy")
ax.set_title("Clean-test accuracy drop")
ax.set_ylim(0, max(0.55, max(values) + 0.1))
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.4f}", ha="center")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "clean_test_accuracy_drop.png", dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
quality_items = [("PSNR", metrics["psnr"]), ("SSIM", metrics["ssim"]), ("L?", metrics["linf"])]
for ax, (name, value) in zip(axes, quality_items):
    ax.bar([name], [value], color="#54A24B")
    ax.set_title(name)
    ax.text(0, value, f"{value:.4f}", ha="center", va="bottom")
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Image quality metrics")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "image_quality_metrics.png", dpi=150)
plt.close(fig)

sample_idx = 0
fig = plot_before_after(clean_x[sample_idx], protected_x[sample_idx], "unlearnable", save=False)
fig.savefig(FIGURE_DIR / "before_after_unlearnable.png", dpi=150)
plt.close(fig)
to_pil = torchvision.transforms.ToPILImage()
num_samples_to_save = min(20, clean_x.size(0))
for idx in range(num_samples_to_save):
    to_pil(clean_x[idx]).save(SAMPLE_DIR / f"original_{idx:03d}.png")
    to_pil(protected_x[idx]).save(SAMPLE_DIR / f"protected_{idx:03d}.png")
# Compatibility copies for existing UI/report paths.
os.makedirs("results/protected_samples", exist_ok=True)
to_pil(clean_x[sample_idx]).save("results/protected_samples/unlearnable_original.png")
to_pil(protected_x[sample_idx]).save("results/protected_samples/unlearnable_protected.png")
print("Saved run to:", RUN_DIR)
print("Key files:")
print("-", TABLE_DIR / "unlearnable_experiment.csv")
print("-", TENSOR_DIR / "protected_dataset.pt")
print("-", MODEL_DIR / "baseline_clean_model.pt")
print("-", MODEL_DIR / "victim_protected_model.pt")


## 4. Optional copy results to Drive

In [ ]:
# Copy local results to Drive results folder if needed.
# Most experiment code writes to ./results first; this keeps a persistent copy in Drive.
if USE_GOOGLE_DRIVE:
    import shutil
    target = Path(RESULTS_ROOT)
    target.mkdir(parents=True, exist_ok=True)
    shutil.copytree("results", target, dirs_exist_ok=True)
    print("Copied local ./results to", target)
